### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)


2. A function or coroutine to execute.

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") #type:ignore

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("why do parrots talk?")
response.content



'Parrots don’t actually "talk" in the human sense—they don’t understand the meaning of words the way we do. Instead, they **mimic sounds** for specific social and survival reasons. Here’s why:\n\n### 1. **Social Bonding**\nParrots are highly social, flock-oriented birds. In the wild, they use vocalizations to:\n- Stay in contact with their flock.\n- Strengthen pair bonds.\n- Greet or acknowledge each other.\nWhen kept as pets, parrots often view their human family as their "flock." Mimicking human speech helps them feel included and bonded to you.\n\n### 2. **Communication & Attention-Seeking**\n- Parrots learn that certain sounds (like their name, "hello," or "I love you") get responses from humans—like talking back, smiling, or giving treats.\n- They quickly associate these sounds with positive attention and repeat them to get that response.\n\n### 3. **Intelligence & Learning**\n- Parrots (especially African Greys, Amazons, and Cockatoos) are among the most intelligent birds.\n- The

In [3]:
from langchain.tools  import tool

@tool
def get_weather(location:str) -> str: 
    """ Get the wether at a location """
    
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What's weather like in Boston")
print(response)

for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool : {tool_call["name"]}")
    print(f"args : {tool_call["args"]}")

content='' additional_kwargs={'tool_calls': [{'id': 'kqn6vpneb', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.068032023, 'completion_tokens_details': None, 'prompt_time': 0.018785176, 'prompt_tokens_details': None, 'queue_time': 0.049598214, 'total_time': 0.086817199}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b83f-2436-7380-b397-37bca5e474da-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'kqn6vpneb', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}
Tool : get_weather
args : {'location': 'Boston'}


### Tool Execution Loops

In [5]:
#  Step 1: Model generates tool calls
message  =[{"role":"user","content":"What's the weather in Boston ?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)


## Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# Step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(message)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."


It's currently sunny in Boston! ☀️


In [6]:
message

[{'role': 'user', 'content': "What's the weather in Boston ?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9ewsxa6p4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.067850242, 'completion_tokens_details': None, 'prompt_time': 0.018876396, 'prompt_tokens_details': None, 'queue_time': 0.053316914, 'total_time': 0.086726638}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b83e-7c3f-75d0-be2d-935cd626b26e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '9ewsxa6p4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}),
 ToolMessage(content="it's sunny